In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pickle
import tauCalc_model as tcm
import importlib
importlib.reload(tcm)

def rest_freq(n):
    '''
    Calculates H transition frequency in MHz
    ------------------
    n            (int)   : Transition level
    ------------------
    return [MHz] (float) : Frequency of n transition
    '''
    return(((1 / (n)**2) - (1 / (n+1)**2)) * 1.1e5 * 3e10 * 1e-6)

def inv_rest_freq(freq):
    '''
    Inverse of rest_freq: estimates n from frequency in MHz
    '''
    # Rearrange the formula approximately numerically
    # rest_freq(n) = ((1/n^2) - (1/(n+1)^2)) * constant
    const = 1.1e5 * 3e10 * 1e-6  # same constant as before
    # We can solve this approximately by iteration or direct approximation
    # For large n, (1/n^2 - 1/(n+1)^2) ≈ 2 / n^3
    # => freq ≈ (2 / n^3) * const  =>  n ≈ (2 * const / freq)^(1/3)
    return (2 * const / freq)**(1/3)

def inv_kms(freq):
    '''
    Converts frequency in MHz to velocity in km/s
    '''
    c = 3e5  # speed of light in km/s
    return (c / freq)

def intDepth(tau, n, V, ff=1):
    return((V/3e5) * rest_freq(n) * tau * 1.064467 * ff)

def beta_n(b, n, T):
    bn = b
    bn_p = np.roll(b, -1)
    n_p = np.roll(n, -1)
    coeff = 3.168 * 1e-6 * n**3 * T * (bn_p - bn) / ((n_p - n) * bn)
    return (1-coeff)

def tau_test(T, EM, n, bbeta):
    chi = 1.58e5 / (T * ((n)**2))
    return(2.046e6 * np.exp(chi)* (T**(-5/2)) * EM * bbeta)

def tau_out(Te, ne, EM = 1e5):
    # Load models
    bn = np.load('models/bn_hrrl_v2.npy')
    beta = np.load('models/beta_hrrl_v2.npy')
    bn_head = pickle.load(open('models/bn_hrrl_head_v2.p', "rb" ))
    

    #Pulling all the T and n values
    Te_arr = np.array(bn_head['Te_v'])
    ne_arr = np.array(bn_head['ne_v'])
    
    #Finding the model index for the data table closest to the chosen temperature and density 
    model_close_id = np.argmin((Te_arr - Te)**2 + (ne_arr - ne)**2)

    print("Te", Te_arr[model_close_id])
    print("ne", ne_arr[model_close_id])
    #Pulling the n values from the bn tables
    n_vals = bn[model_close_id, 0, :]
    
    #Choosing the first set of bn values from the model
    bn_vals = bn[model_close_id, 1, :]
    betan_vals = beta_n(bn_vals, n_vals, Te)

    tau_calc = tau_test(Te, EM, n_vals, bn_vals*betan_vals)

    return(n_vals, tau_calc)
def tau_C(Te, n, EM):
    #Formula from blue book eq 2.95
    return(0.08235 * Te**(-1.35) * (rest_freq(n)/1000)**(-2.1) * EM)

In [ ]:
mark = "x"
reference_points = [
    {"name": "Anantharamaiah+ 1993", "tau": -1.4/64.4, "n": 92, "V": 170, "ff":1, "marker": mark, "color": "magenta"},
    {"name": None,                   "tau": -0.47/88.8, "n": 92, "V": 350, "ff":1, "marker": mark, "color": "magenta"},
    {"name": None,                   "tau": -0.99/48.3, "n": 92, "V": 310, "ff":1, "marker": mark, "color": "magenta"},
    {"name": "Mohan+ 2001",          "tau": -0.53/28, "n": 92, "V": 95, "ff":1, "marker": mark, "color": "darkturquoise"},
    {"name": "Phookun+ 1998",        "tau": -0.5/58.9, "n": 92, "V": 377, "ff":1, "marker": mark, "color": "gold"}, #unclear, method doesn't reproduce 0.65? #good example
    {"name": "Roy+ 2005",            "tau": -0.56/49, "n": 92, "V": 114, "ff":1, "marker": mark, "color": "teal"}, #unclear (0.56/49 value)
    {"name": "Zhao+ 1996",           "tau": -0.8/75, "n": 92, "V": 95, "ff":1, "marker": mark, "color": "midnightblue"}, #unclear which model (chose D)
    {"name": "Mohan+ 2005",          "tau": -1/40, "n": 92, "V": 190, "ff":1, "marker": mark, "color": "crimson"}, #unclear which Line Flux, FWHM, Filling Factor
    {"name": None,                   "tau": -1.6/37, "n": 75, "V": 230, "ff":1, "marker": mark, "color": "crimson"}, #unclear which Filling Factor
    {"name": None,                   "tau": -1.8/675, "n": 166, "V": 160, "ff":1, "marker": mark, "color": "crimson"}, #unclear which Filling Factor
]
def plot_reference_points(ref_points, intDepth_func, ax, axins):
    """
    Plots reference points on an existing matplotlib figure.
    ----------------------------
    Parameters:
        ref_points (list of dict): Each dict should have keys:
            - 'n' (int)
            - 'marker' (str)
            - 'color' (str)
            - and either:
              - 'tau' (float) and 'V' (float), or
              - 'coeff' (float)
            - 'name' (str or None)
        tau_out (np.array): Used for normalizing the coefficient.
        intDepth_func (callable): Function of (tau, n, V) to compute coefficient.
    """
    
    for ref in ref_points:
        coeff = intDepth_func(ref["tau"], ref["n"], ref["V"], ref["ff"])

        # White halo underneath
        axins.scatter(
            ref["n"], -coeff*1e6,
            marker=ref["marker"],
            c="white",
            s=70,
            linewidths=4,
            zorder=9
        )

        # Actual observational point
        axins.scatter(
            ref["n"], -coeff*1e6,
            marker=ref["marker"],
            c=ref["color"],
            s=35,
            linewidths=2,
            zorder=10,
            label=ref["name"] if ref["name"] else None
        )

In [ ]:
n_vals, tau1 = tau_out(8000, 1e1, 1e7)
n_vals, tau2 = tau_out(8000, 1e3, 1e6)
n_vals, tau3 = tau_out(8000, 1e4, 1e7)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from sympy import Line2D
mpl.rcParams['figure.dpi'] = 300
import numpy as np
#from RRL_Code import tauCalc_model as tcm
import tauCalc_model as tcm
import importlib
importlib.reload(tcm)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -- Main figure and axis --
fig, axins = plt.subplots(figsize=(8, 6))

#axins = inset_axes(ax, width="45%", height="45%", loc='lower left', 
#                   bbox_to_anchor=(0.07, 0.07, 0.88, 0.9), bbox_transform=ax.transAxes)

#axins.loglog(n_vals, abs(tau_vals_smooth), c = '#DB3A34')
#axins.loglog(n_vals, np.abs(tau1), c = '#DB3A34')
#axins.loglog(n_vals, np.abs(tau2), c = '#177E89')
#axins.loglog(n_vals, np.abs(tau3), c = '#837099')
#axins.loglog(n_vals, abs(-np.ones_like(n_vals)*0.1*30/3e5 * rest_freq(n_vals)*1e6), c = '#084C61')
#_, tau1 = tcm.tau_out(8000, 1e1, 1e7)
n_vals = np.arange(30, 601, 1)
Te = 8000
#print(n_vals)
#tau_continuum = [np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))]
tau_continuum = [np.ones_like(n_vals), np.concatenate((np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))))]
tau_continuum = [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e7))]
print(len(tau_continuum))
lineweigh = 3


n_vals, plotbig = np.abs(tcm.tau_out(Te, 1e4, 1e7))# * tau_continuum#*(np.exp(-tau_C(Te, n_vals, 1e7)))

#axins.loglog(*np.abs(tcm.tau_out(Te, 1e1, 1e4)) * [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e4))], label = r'$n_e = 10, f~\rm EM = 10^4$', c = 'crimson', lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e5))* [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e5))], label = r'$n_e = 10^3, f~\rm EM = 10^5$', c = 'saddlebrown', lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e6))* [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e6))], label = r'$n_e = 10^3, f~\rm EM = 10^6$', c = '#177E89', lw = lineweigh)
#axins.loglog(n_vals[:165], plotbig[:165], label = r'$n_e = 10^4, f~\rm EM = 10^7$', c = 'goldenrod', lw = lineweigh)
#axins.loglog(n_vals[164:], plotbig[164:], ls = '--', c = 'goldenrod', lw = lineweigh)

x_10, y_10 = np.abs(tcm.tau_out(Te, 1e1, 1e4))
N_cut = 63
axins.loglog(n_vals, np.abs(-np.ones_like(n_vals)*0.1*20/3e5*rest_freq(n_vals)*1e6), label = r'PO11 (H)', c = '#837099', lw = lineweigh)
axins.loglog(x_10[N_cut:], y_10[N_cut:], label = r'Model A (H)', c = 'crimson', lw = lineweigh)
axins.loglog(x_10[:N_cut+1], y_10[:N_cut+1], c = 'crimson', lw = lineweigh, ls = '--')

axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e5)), label = r'Model B (H)', c = 'saddlebrown', lw = lineweigh)
axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e6)), label = r'Model C (H)', c = '#177E89', lw = lineweigh)
n_modelD, modelD_split = np.abs(tcm.tau_out(Te, 1e4, 1e7))
modelD_cut = 167
axins.loglog(n_modelD[:modelD_cut], modelD_split[:modelD_cut], ls = '-',label = r'Model D (H)', c = 'goldenrod', lw = lineweigh)
axins.loglog(n_modelD[modelD_cut:], modelD_split[modelD_cut:], ls = '--', c = 'goldenrod', lw = lineweigh)

#axins.loglog(n_vals[:170], np.absolute(tau_model_carbon)[:170], label = r'Carbon', ls = '--', c = "#000000", lw = 2.5)
#axins.loglog(n_vals[170:], np.absolute(tau_model_carbon)[170:-1], label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
#data_c = np.loadtxt("carbon_tau_combined.txt")
#full_n_combined = data_c[:, 0]
#full_tau_combined = data_c[:, 1]
#axins.loglog(full_n_combined, np.absolute(full_tau_combined), label = r'Carbon (combined)', ls = '-', c = "#000000", lw = 2.5)

N_cut2 = 155
N_spaced = 47
n_vals_carbon, tau_carbon_2 = (tcm.tau_out(100, 0.05, 0.02))
axins.loglog(n_vals_carbon[:N_cut2], tau_carbon_2[:N_cut2], ls = '--', c = "#FF46E0", lw = lineweigh)
axins.loglog(n_vals_carbon[N_cut2:N_cut2+N_spaced], np.abs(tau_carbon_2[N_cut2:N_cut2+N_spaced]), label = r'Model E (C)', ls = '-', c = "#FF46E0", lw = lineweigh)
#axins.loglog(n_vals_carbon, tau_carbon_2, label = r'Model E', ls = '--', c = "#FF46E0", lw = lineweigh)
axins.loglog(n_vals_carbon[N_cut2+N_spaced:], tau_carbon_2[N_cut2+N_spaced:], ls = '--', c = "#FF46E0", lw = lineweigh)
#plt.plot(*(tcm.tau_out(100, 0.05, 0.02)))
#axins.loglog(*np.abs(tcm.tau_out(85, 1e-3, 0.02, i_n = 2)), label = r'Model E', ls = '-', c = "#FF46E0", lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(85, 1e-2, 0.02, i_n = 4)), label = r'Carbon ($n_e = 10^{-2}$)', ls = '-.', c = "#AEAEAE", lw = 2.5)
#axins.loglog(*np.abs(tcm.tau_out(85, 0.04, 0.02, i_n = 6)), label = r'Carbon ($n_e = 4 \times 10^{-2}$)', ls = '--', c = "#000000", lw = 2.5)
#axins.loglog(*np.abs(tcm.tau_out(85, 0.1, 0.02, i_n = 8)), label = r'Carbon ($n_e = 10^{-1}$)', ls = '-.', c = "#BF40FF", lw = 2.5)


#axins.loglog(*np.abs(tcm.tau_out(100, 1e-2, 0.02)), label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
#print(np.abs(tcm.tau_out(10000, 10, 0.01)[1]))

#axins.set_xscale('linear')
#axins.set_yscale('linear')
#axins.loglog(n_cut_block, abs(smoothing(n_cut_block, results, 190)), c = '#FFBF00')
# Set axis limits
axins.set_xlim(80,300)
axins.set_ylim(1e-2, 100)

# Set specific ticks and format them as integers
xticks = [100, 125, 150, 175, 200, 250, 300, 400]
axins.set_xticks(xticks)
plt.minorticks_off()
axins.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))

yticks = [1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
axins.set_yticks(yticks)
axins.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"$-10^{{{int(np.log10(y))}}}$"))
#axins.spines['left'].set_linestyle(':')
#axins.spines['left'].set_color('#157F1F')
#axins.spines['left'].set_linewidth(1.5)

axins.axvline(86, ls = ':', c = "#000000", lw = 2.5)
axins.axvline(166, ls = ':', c = "#000000", lw = 2.5)
axins.axvline(240, ls = ':', c = "#000000", lw = 2.5)
axins.set_ylim(1e-1, 1e6)
#Shaver77coeff = intDepth(-0.011, 166, 150)*1e6
#axins.scatter(166, -Shaver77coeff, marker= "^", c= 'darkturquoise', zorder = 3)
#emi = -14.8 #(-14.8 / (20 * 288*1e6 / 3e6))
#print(Shaver77coeff, 'shaver')

axins.tick_params(labelsize=12)
axins.set_visible(True)
ax = 0
plot_reference_points(reference_points, intDepth, ax, axins)
emigpt = axins.scatter(288, 14.8, marker= "*", c= 'black', zorder = 10, label = 'Emig+ 2019')
emigpt2 = axins.scatter(144, 328.127, marker= "*", c= 'maroon', zorder = 10, label = 'Emig+ 2023')

axins.errorbar(
    288, 14.8, 
    yerr=10,              # length of the arrow
    lolims=True,           # arrow pointing upward
    fmt="none",            # don't add another point marker
    color="black",
    zorder=2
)
#-328.12706434139113

handles, labels = axins.get_legend_handles_labels()

# Split into curves (lines without markers) and points (everything else)
line_handles = [h for h in handles if isinstance(h, plt.Line2D) and h.get_marker() == 'None']
line_labels  = [l for h, l in zip(handles, labels) if isinstance(h, plt.Line2D) and h.get_marker() == 'None']

scatter_handles = [h for h in handles if not (isinstance(h, plt.Line2D) and h.get_marker() == 'None')]
scatter_labels  = [l for h, l in zip(handles, labels) if not (isinstance(h, plt.Line2D) and h.get_marker() == 'None')]

# Two legends
legend1 = axins.legend(line_handles, line_labels, loc='upper right', fontsize=10, framealpha = 1)# title=r"$\tau_{RRL}\left(n_e, f~\rm EM\right)$ Models", )
#bbox_to_anchor = (0.52, 0.32),
#legend2 = axins.legend(scatter_handles, scatter_labels, loc='lower left', title="Observations", fontsize=10)
from matplotlib.lines import Line2D

second_legend = [
    Line2D([], [], marker='x', linestyle='None',
           color='k', markersize=6, label='Local'),
    Line2D([], [], marker='*', linestyle='None',
           color='k', markersize=6, label='Cosmological ($z > 0.5$)')
]

leg2 = axins.legend(handles=second_legend, bbox_to_anchor=(0.427, 0.82), title="RRL Observations", loc='lower left', title_fontsize=12, framealpha = 1)

axins.add_artist(legend1)  # re-add first legend so both show

axins.set_xlabel(r"Quantum Number ($n$)", fontsize = 12)
axins.set_ylabel(r"$e^{-\tau_C}\int \tau_{RRL}(\nu)~d\nu$ [Hz]", fontsize = 12)
axins.set_ylabel(r"$\mathcal{T}(n_e,f~\rm{EM})$ [Hz]", fontsize = 12)
#axins.set_ylabel(r"$\int \tau_{\rm RRL}(\nu)\, e^{-\tau_C(\nu)}\, d\nu \; [\mathrm{Hz}]$", fontsize = 12)
axins.tick_params(right = True)
secax = axins.secondary_xaxis('top', functions=(rest_freq, inv_rest_freq))
xticks_2 = [400, 800, 1400, 2000, 4000, 10e3]
#xticks_2 = rest_freq(np.array(xticks))
secax.set_xticks(xticks_2)
secax.minorticks_off()
secax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
secax.set_xlabel("Rest Frequency [MHz]")

#plt.savefig("tau_Plot2.png", dpi=300, bbox_inches='tight')

axins.text(86-4, 5e2, r'$n=86$', rotation=90, va='top', ha='left')
axins.text(166-7, 1e2, r'$n=166$', rotation=90, va='top', ha='left')
axins.text(238-8, 2e1, r'$n=240$', rotation=90, va='top', ha='left')

axins.tick_params(axis='both', colors='black')
secax.tick_params(axis='both', colors='black')

axins.xaxis.label.set_color('black')
axins.yaxis.label.set_color('black')
secax.xaxis.label.set_color('black')

for spine in axins.spines.values():
    spine.set_color('black')
for spine in secax.spines.values():
    spine.set_color('black')

# Make figure + axes transparent
#fig.patch.set_alpha(0)
axins.set_facecolor('none')
secax.set_facecolor('none')

# --- Secondary Y Axis: Velocity (km/s) ---
c_kms = 3e5
nu_ref = 1400  # MHz (set appropriately)

def hz_to_kms(dnu):
    return dnu / 1e6 * c_kms / nu_ref

def kms_to_hz(v):
    return v * nu_ref / c_kms * 1e6

secay = axins.secondary_yaxis('right', functions=(hz_to_kms, kms_to_hz))
secay.set_ylabel("$\Delta V_\mathrm{EW}$ [km/s]", fontsize=12)

secay.tick_params(axis='y', colors='black')
secay.yaxis.label.set_color('black')
for spine in secay.spines.values():
    spine.set_color('black')
#plt.savefig("figures/tau_Obs_plot_Carbon.png", dpi=300, bbox_inches='tight')

In [ ]:
plt.loglog(*(tcm.tau_out(Te, 1e1, 1e4)), label = 'Model A')
plt.loglog(*(tcm.tau_out(Te, 1e4, 1e7)), label = 'Model D')
plt.loglog(*(tcm.tau_out(Te, 1e3, 1e5)), label = r'Model B', c = 'saddlebrown', lw = lineweigh)
plt.loglog(*(tcm.tau_out(Te, 1e3, 1e6)), label = r'Model C', c = '#177E89', lw = lineweigh)
plt.loglog(*(tcm.tau_out(100, 0.05, 0.02)), label = 'carbon')
#print(((tcm.tau_out(Te, 1e4, 1e7)[:50][:50])))
plt.xlim(80,300)
plt.ylim(1e-2, 1e6)
plt.legend()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from sympy import Line2D
mpl.rcParams['figure.dpi'] = 300
import numpy as np
#from RRL_Code import tauCalc_model as tcm
import tauCalc_model as tcm
import importlib
importlib.reload(tcm)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -- Main figure and axis --
fig, axins = plt.subplots(figsize=(8, 6))

#axins = inset_axes(ax, width="45%", height="45%", loc='lower left', 
#                   bbox_to_anchor=(0.07, 0.07, 0.88, 0.9), bbox_transform=ax.transAxes)

#axins.loglog(n_vals, abs(tau_vals_smooth), c = '#DB3A34')
#axins.loglog(n_vals, np.abs(tau1), c = '#DB3A34')
#axins.loglog(n_vals, np.abs(tau2), c = '#177E89')
#axins.loglog(n_vals, np.abs(tau3), c = '#837099')
#axins.loglog(n_vals, abs(-np.ones_like(n_vals)*0.1*30/3e5 * rest_freq(n_vals)*1e6), c = '#084C61')
#_, tau1 = tcm.tau_out(8000, 1e1, 1e7)
n_vals = np.arange(30, 601, 1)
Te = 8000
#print(n_vals)
#tau_continuum = [np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))]
tau_continuum = [np.ones_like(n_vals), np.concatenate((np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))))]
tau_continuum = [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e7))]
print(len(tau_continuum))
lineweigh = 3


n_vals, plotbig = np.abs(tcm.tau_out(Te, 1e4, 1e7))# * tau_continuum#*(np.exp(-tau_C(Te, n_vals, 1e7)))

#axins.loglog(*np.abs(tcm.tau_out(Te, 1e1, 1e4)) * [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e4))], label = r'$n_e = 10, f~\rm EM = 10^4$', c = 'crimson', lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e5))* [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e5))], label = r'$n_e = 10^3, f~\rm EM = 10^5$', c = 'saddlebrown', lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e6))* [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e6))], label = r'$n_e = 10^3, f~\rm EM = 10^6$', c = '#177E89', lw = lineweigh)
#axins.loglog(n_vals[:165], plotbig[:165], label = r'$n_e = 10^4, f~\rm EM = 10^7$', c = 'goldenrod', lw = lineweigh)
#axins.loglog(n_vals[164:], plotbig[164:], ls = '--', c = 'goldenrod', lw = lineweigh)

x_10, y_10 = np.abs(tcm.tau_out(Te, 1e1, 1e4))
N_cut = 63
axins.loglog(n_vals, np.abs(-np.ones_like(n_vals)*0.1*20/3e5*rest_freq(n_vals)*1e6), label = r'PO11', c = '#837099', lw = lineweigh)

'''
axins.loglog(x_10[N_cut:], y_10[N_cut:], label = r'Model A', c = 'crimson', lw = lineweigh)
axins.loglog(x_10[:N_cut+1], y_10[:N_cut+1], c = 'crimson', lw = lineweigh, ls = '--')

axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e5)), label = r'Model B', c = 'saddlebrown', lw = lineweigh)
axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e6)), label = r'Model C', c = '#177E89', lw = lineweigh)
n_modelD, modelD_split = np.abs(tcm.tau_out(Te, 1e4, 1e7))
modelD_cut = 167
axins.loglog(n_modelD[:modelD_cut], modelD_split[:modelD_cut], ls = '-',label = r'Model D', c = 'goldenrod', lw = lineweigh)
axins.loglog(n_modelD[modelD_cut:], modelD_split[modelD_cut:], ls = '--', c = 'goldenrod', lw = lineweigh)

N_cut2 = 155
N_spaced = 47
n_vals_carbon, tau_carbon_2 = (tcm.tau_out(100, 0.05, 0.02))
axins.loglog(n_vals_carbon[:N_cut2], tau_carbon_2[:N_cut2], ls = '--', c = "#FF46E0", lw = lineweigh)
axins.loglog(n_vals_carbon[N_cut2:N_cut2+N_spaced], np.abs(tau_carbon_2[N_cut2:N_cut2+N_spaced]), label = r'Model E', ls = '-', c = "#FF46E0", lw = lineweigh)
axins.loglog(n_vals_carbon[N_cut2+N_spaced:], tau_carbon_2[N_cut2+N_spaced:], ls = '--', c = "#FF46E0", lw = lineweigh)
'''



#axins.loglog(n_vals[:170], np.absolute(tau_model_carbon)[:170], label = r'Carbon', ls = '--', c = "#000000", lw = 2.5)
#axins.loglog(n_vals[170:], np.absolute(tau_model_carbon)[170:-1], label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
#data_c = np.loadtxt("carbon_tau_combined.txt")
#full_n_combined = data_c[:, 0]
#full_tau_combined = data_c[:, 1]
#axins.loglog(full_n_combined, np.absolute(full_tau_combined), label = r'Carbon (combined)', ls = '-', c = "#000000", lw = 2.5)

#axins.loglog(n_vals_carbon, tau_carbon_2, label = r'Model E', ls = '--', c = "#FF46E0", lw = lineweigh)
#plt.plot(*(tcm.tau_out(100, 0.05, 0.02)))
#axins.loglog(*np.abs(tcm.tau_out(85, 1e-3, 0.02, i_n = 2)), label = r'Model E', ls = '-', c = "#FF46E0", lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(85, 1e-2, 0.02, i_n = 4)), label = r'Carbon ($n_e = 10^{-2}$)', ls = '-.', c = "#AEAEAE", lw = 2.5)
#axins.loglog(*np.abs(tcm.tau_out(85, 0.04, 0.02, i_n = 6)), label = r'Carbon ($n_e = 4 \times 10^{-2}$)', ls = '--', c = "#000000", lw = 2.5)
#axins.loglog(*np.abs(tcm.tau_out(85, 0.1, 0.02, i_n = 8)), label = r'Carbon ($n_e = 10^{-1}$)', ls = '-.', c = "#BF40FF", lw = 2.5)


#axins.loglog(*np.abs(tcm.tau_out(100, 1e-2, 0.02)), label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
#print(np.abs(tcm.tau_out(10000, 10, 0.01)[1]))

#axins.set_xscale('linear')
#axins.set_yscale('linear')
#axins.loglog(n_cut_block, abs(smoothing(n_cut_block, results, 190)), c = '#FFBF00')
# Set axis limits
axins.set_xlim(80,300)
axins.set_ylim(1e-2, 100)

# Set specific ticks and format them as integers
xticks = [100, 125, 150, 175, 200, 250, 300, 400]
axins.set_xticks(xticks)
plt.minorticks_off()
axins.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))

yticks = [1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
axins.set_yticks(yticks)
axins.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"$-10^{{{int(np.log10(y))}}}$"))
#axins.spines['left'].set_linestyle(':')
#axins.spines['left'].set_color('#157F1F')
#axins.spines['left'].set_linewidth(1.5)

axins.axvline(86, ls = ':', c = "#FFFFFF", lw = 2.5)
axins.axvline(166, ls = ':', c = "#FFFFFF", lw = 2.5)
axins.axvline(240, ls = ':', c = "#FFFFFF", lw = 2.5)
axins.set_ylim(1e-1, 1e6)
#Shaver77coeff = intDepth(-0.011, 166, 150)*1e6
#axins.scatter(166, -Shaver77coeff, marker= "^", c= 'darkturquoise', zorder = 3)
#emi = -14.8 #(-14.8 / (20 * 288*1e6 / 3e6))
#print(Shaver77coeff, 'shaver')

axins.tick_params(labelsize=12)
axins.set_visible(True)
ax = 0
plot_reference_points(reference_points, intDepth, ax, axins)
emigpt = axins.scatter(288, 14.8, marker= "*", c= 'white', zorder = 3, label = 'Emig+ 2019')
emigpt2 = axins.scatter(144, 328.127, marker= "*", c= "#FFC3C3", zorder = 3, label = 'Emig+ 2023')

axins.errorbar(
    288, 14.8, 
    yerr=10,              # length of the arrow
    lolims=True,           # arrow pointing upward
    fmt="none",            # don't add another point marker
    color="white",
    zorder=2
)
#-328.12706434139113

handles, labels = axins.get_legend_handles_labels()

# Split into curves (lines without markers) and points (everything else)
line_handles = [h for h in handles if isinstance(h, plt.Line2D) and h.get_marker() == 'None']
line_labels  = [l for h, l in zip(handles, labels) if isinstance(h, plt.Line2D) and h.get_marker() == 'None']

scatter_handles = [h for h in handles if not (isinstance(h, plt.Line2D) and h.get_marker() == 'None')]
scatter_labels  = [l for h, l in zip(handles, labels) if not (isinstance(h, plt.Line2D) and h.get_marker() == 'None')]

# Two legends
legend1 = axins.legend(line_handles, line_labels, loc='upper right', fontsize=10, framealpha = 1)# title=r"$\tau_{RRL}\left(n_e, f~\rm EM\right)$ Models", )
#bbox_to_anchor = (0.52, 0.32),
#legend2 = axins.legend(scatter_handles, scatter_labels, loc='lower left', title="Observations", fontsize=10)
from matplotlib.lines import Line2D

second_legend = [
    Line2D([], [], marker='x', linestyle='None',
           color='k', markersize=6, label='Local'),
    Line2D([], [], marker='*', linestyle='None',
           color='k', markersize=6, label='Cosmological ($z > 0.5$)')
]

leg2 = axins.legend(handles=second_legend, bbox_to_anchor=(0.467, 0.82), title="RRL Observations", loc='lower left', title_fontsize=12, framealpha = 1)

axins.add_artist(legend1)  # re-add first legend so both show

axins.set_xlabel(r"Quantum Number ($n$)", fontsize = 12)
axins.set_ylabel(r"$e^{-\tau_C}\int \tau_{RRL}(\nu)~d\nu$ [Hz]", fontsize = 12)
axins.set_ylabel(r"$\mathcal{T}(n_e,f~\rm{EM})$ [Hz]", fontsize = 12)
#axins.set_ylabel(r"$\int \tau_{\rm RRL}(\nu)\, e^{-\tau_C(\nu)}\, d\nu \; [\mathrm{Hz}]$", fontsize = 12)
axins.tick_params(right = True)
secax = axins.secondary_xaxis('top', functions=(rest_freq, inv_rest_freq))
xticks_2 = [400, 800, 1400, 2000, 4000, 10e3]
#xticks_2 = rest_freq(np.array(xticks))
secax.set_xticks(xticks_2)
secax.minorticks_off()
secax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
secax.set_xlabel("Rest Frequency [MHz]")

#plt.savefig("tau_Plot2.png", dpi=300, bbox_inches='tight')

axins.tick_params(axis='both', colors='white')
secax.tick_params(axis='both', colors='white')

axins.xaxis.label.set_color('white')
axins.yaxis.label.set_color('white')
secax.xaxis.label.set_color('white')

for spine in axins.spines.values():
    spine.set_color('white')
for spine in secax.spines.values():
    spine.set_color('white')

# Make figure + axes transparent
#fig.patch.set_alpha(0)
axins.set_facecolor('none')
secax.set_facecolor('none')

# --- Secondary Y Axis: Velocity (km/s) ---
c_kms = 3e5
nu_ref = 1400  # MHz (set appropriately)

def hz_to_kms(dnu):
    return dnu / 1e6 * c_kms / nu_ref

def kms_to_hz(v):
    return v * nu_ref / c_kms * 1e6

secay = axins.secondary_yaxis('right', functions=(hz_to_kms, kms_to_hz))
secay.set_ylabel("$\Delta V_\mathrm{EW}$ [km/s]", fontsize=12)

secay.tick_params(axis='y', colors='white')
secay.yaxis.label.set_color('white')
for spine in secay.spines.values():
    spine.set_color('white')

fig.patch.set_alpha(0)
axins.set_facecolor('none')
secax.set_facecolor('none')
secay.set_facecolor('none')
#plt.savefig("figures/tau_Obs_plot_Carbon.png", dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300
import numpy as np
#from RRL_Code import tauCalc_model as tcm
import tauCalc_model as tcm
import importlib
importlib.reload(tcm)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -- Main figure and axis --
fig, axins = plt.subplots(figsize=(8, 6))

#axins = inset_axes(ax, width="45%", height="45%", loc='lower left', 
#                   bbox_to_anchor=(0.07, 0.07, 0.88, 0.9), bbox_transform=ax.transAxes)

#axins.loglog(n_vals, abs(tau_vals_smooth), c = '#DB3A34')
#axins.loglog(n_vals, np.abs(tau1), c = '#DB3A34')
#axins.loglog(n_vals, np.abs(tau2), c = '#177E89')
#axins.loglog(n_vals, np.abs(tau3), c = '#837099')
#axins.loglog(n_vals, abs(-np.ones_like(n_vals)*0.1*30/3e5 * rest_freq(n_vals)*1e6), c = '#084C61')
#_, tau1 = tcm.tau_out(8000, 1e1, 1e7)
n_vals = np.arange(30, 601, 1)
Te = 8000
#print(n_vals)
#tau_continuum = [np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))]
tau_continuum = [np.ones_like(n_vals), np.concatenate((np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))))]
tau_continuum = [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e7))]
print(len(tau_continuum))
lineweigh = 3


n_vals, plotbig = np.abs(tcm.tau_out(Te, 1e4, 1e7))# * tau_continuum#*(np.exp(-tau_C(Te, n_vals, 1e7)))

#axins.loglog(*np.abs(tcm.tau_out(Te, 1e1, 1e4)) * [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e4))], label = r'$n_e = 10, f~\rm EM = 10^4$', c = 'crimson', lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e5))* [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e5))], label = r'$n_e = 10^3, f~\rm EM = 10^5$', c = 'saddlebrown', lw = lineweigh)
#axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e6))* [np.ones_like(n_vals),  np.exp(-tau_C(Te, n_vals, 1e6))], label = r'$n_e = 10^3, f~\rm EM = 10^6$', c = '#177E89', lw = lineweigh)
#axins.loglog(n_vals[:165], plotbig[:165], label = r'$n_e = 10^4, f~\rm EM = 10^7$', c = 'goldenrod', lw = lineweigh)
#axins.loglog(n_vals[164:], plotbig[164:], ls = '--', c = 'goldenrod', lw = lineweigh)

x_10, y_10 = np.abs(tcm.tau_out(Te, 1e1, 1e4))
N_cut = 63
axins.loglog(x_10[N_cut:], y_10[N_cut:], label = r'$n_e = 10, f~\rm EM = 10^4$', c = 'crimson', lw = lineweigh)
axins.loglog(x_10[:N_cut+1], y_10[:N_cut+1], c = 'crimson', lw = lineweigh, ls = '--')

axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e5)), label = r'$n_e = 10^3, f~\rm EM = 10^5$', c = 'saddlebrown', lw = lineweigh)
axins.loglog(*np.abs(tcm.tau_out(Te, 1e3, 1e6)), label = r'$n_e = 10^3, f~\rm EM = 10^6$', c = '#177E89', lw = lineweigh)
axins.loglog(*np.abs(tcm.tau_out(Te, 1e4, 1e7)), label = r'$n_e = 10^4, f~\rm EM = 10^7$', c = 'goldenrod', lw = lineweigh)
axins.loglog(n_vals, np.abs(-np.ones_like(n_vals)*0.1*30/3e5*rest_freq(n_vals)*1e6), label = r'$\tau_{RRL}=0.1$ (PO11)', c = '#837099', lw = lineweigh)
#axins.loglog(n_cut_block, abs(smoothing(n_cut_block, results, 190)), c = '#FFBF00')
# Set axis limits
axins.set_xlim(80,300)
axins.set_ylim(1e-2, 100)

# Set specific ticks and format them as integers
xticks = [100, 125, 150, 175, 200, 300]
axins.set_xticks(xticks)
plt.minorticks_off()
axins.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))

yticks = [1e0, 1e2, 1e4, 1e5, 1e6]
axins.set_yticks(yticks)
axins.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"$-10^{{{int(np.log10(y))}}}$"))
#axins.spines['left'].set_linestyle(':')
#axins.spines['left'].set_color('#157F1F')
#axins.spines['left'].set_linewidth(1.5)

axins.axvline(86, ls = ':', c = "#FFFFFF", lw = 2.5)
axins.axvline(166, ls = ':', c = "#FFFFFF", lw = 2.5)
axins.axvline(240, ls = ':', c = "#FFFFFF", lw = 2.5)
axins.set_ylim(1, 1e6)
#Shaver77coeff = intDepth(-0.011, 166, 150)*1e6
#axins.scatter(166, -Shaver77coeff, marker= "^", c= 'darkturquoise', zorder = 3)
#emi = -14.8 #(-14.8 / (20 * 288*1e6 / 3e6))
#print(Shaver77coeff, 'shaver')

axins.tick_params(labelsize=12)
axins.set_visible(True)
ax = 0
plot_reference_points(reference_points, intDepth, ax, axins)
emigpt = axins.scatter(288, 14.8, marker= "*", c= 'white', zorder = 3, label = 'Emig+ 2019')
emigpt2 = axins.scatter(144, 328.127, marker= "*", c= 'coral', zorder = 3, label = 'Emig+ 2023')

axins.errorbar(
    288, 14.8, 
    yerr=10,              # length of the arrow
    lolims=True,           # arrow pointing upward
    fmt="none",            # don't add another point marker
    color="white",
    zorder=2
)
#-328.12706434139113

handles, labels = axins.get_legend_handles_labels()

# Split into curves (lines without markers) and points (everything else)
line_handles = [h for h in handles if isinstance(h, plt.Line2D) and h.get_marker() == 'None']
line_labels  = [l for h, l in zip(handles, labels) if isinstance(h, plt.Line2D) and h.get_marker() == 'None']

scatter_handles = [h for h in handles if not (isinstance(h, plt.Line2D) and h.get_marker() == 'None')]
scatter_labels  = [l for h, l in zip(handles, labels) if not (isinstance(h, plt.Line2D) and h.get_marker() == 'None')]

# Two legends
legend1 = axins.legend(line_handles, line_labels, loc='upper right', title=r"$\tau_{RRL}\left(n_e, f~\rm EM\right)$ Models", fontsize=10)
legend2 = axins.legend(scatter_handles, scatter_labels, loc='lower left', title="Observations", fontsize=10)

axins.add_artist(legend1)  # re-add first legend so both show

axins.set_xlabel(r"Quantum Number ($n$)", fontsize = 12)
axins.set_ylabel(r"$e^{-\tau_C}\int \tau_{RRL}(\nu)~d\nu$ [Hz]", fontsize = 12)
#axins.set_ylabel(r"$\mathcal{T}(n_e,f~\rm{EM})$ [Hz]", fontsize = 12)
axins.set_ylabel(r"$e^{-\tau_C(\nu)}\, \int \tau_{\rm RRL}(\nu)\, d\nu \; [\mathrm{Hz}]$", fontsize = 12)
axins.tick_params(right = True)
secax = axins.secondary_xaxis('top', functions=(rest_freq, inv_rest_freq))
xticks_2 = [400, 800, 1400, 2000, 4000, 10e3]
#xticks_2 = rest_freq(np.array(xticks))
secax.set_xticks(xticks_2)
secax.minorticks_off()
secax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
secax.set_xlabel("Rest Frequency [MHz]")

#

axins.tick_params(axis='both', colors='white')
secax.tick_params(axis='both', colors='white')

axins.xaxis.label.set_color('white')
axins.yaxis.label.set_color('white')
secax.xaxis.label.set_color('white')

# --- Secondary Y Axis: Velocity (km/s) ---
c_kms = 3e5
nu_ref = 1400  # MHz (set appropriately)

def hz_to_kms(dnu):
    return dnu / 1e6 * c_kms / nu_ref

def kms_to_hz(v):
    return v * nu_ref / c_kms * 1e6

secay = axins.secondary_yaxis('right', functions=(hz_to_kms, kms_to_hz))
secay.set_ylabel("$\Delta$V [km/s]", fontsize=12)

secay.tick_params(axis='y', colors='white')
secay.yaxis.label.set_color('white')

for spine in secay.spines.values():
    spine.set_color('white')

for spine in axins.spines.values():
    spine.set_color('white')
for spine in secax.spines.values():
    spine.set_color('white')

# Make figure + axes transparent
fig.patch.set_alpha(0)
axins.set_facecolor('none')
secax.set_facecolor('none')
secay.set_facecolor('none')
#plt.savefig("figures/black/testfig.png", dpi=300, bbox_inches='tight')

In [ ]:
#tau_continuum = np.concatenate((np.ones(136), np.exp(-tau_C(Te, n_vals[136:], 1e7))))
#plt.plot(n_vals, tau_continuum)
n_vals = np.arange(30, 601, 1)
#print(n_vals)
tau_continuum = tau_C(Te, n_vals, 1e7)
plt.plot(n_vals, np.exp(-tau_continuum))
plt.axvline(166)


In [ ]:
importlib.reload(tcm)
#n_test, tautest = tcm.tau_out(85, 1e-2, 0.02)
#plt.plot(n_test, tautest, label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
plt.plot(n_vals, tau_model_carbon[:-1], label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
#obs1 = intDepth(5e-4, 626, 3.3, ff=1)
#obs2 = intDepth(5e-4, 726, 4.7, ff=1)
#obs3 = intDepth(5e-4, 826, 3.7, ff=1)
#plt.scatter(626, -obs1)
#plt.scatter(726, -obs2)
#plt.scatter(826, -obs3)
#plt.scatter(260, -8.54)
#plt.scatter(260, -4.85)
#plt.scatter(266, -8.27)
#plt.scatter(266, -4.06)
#plt.scatter(271, -8.48)
#plt.scatter(271, -4.23)
tau_array2 = [-8.54, -8.27, -8.48, -7.6]
n_array2 = [260, 266, 271, 276]
plt.scatter(n_array2, tau_array2)
#plt.scatter(276, -7.6)
#tau_array = np.array([5.63, 1.75, 5.58, 1.97, 6.15, 2.89, ])
#doi:10.1093/mnras/stw2818
tau_array = [5.63, 5.58, 6.15, 6.61, 6.81, 7.3, 7.47, 7.97, 8.82, 8.74, 7.95, 9.05]
n_array = [438, 448, 459, 467, 477, 485, 496, 510, 527, 542, 559, 575]
plt.scatter(n_array, np.array(tau_array))
#plt.scatter(276, -3.68)
#plt.ylim(1e-10, 1e3)

In [ ]:
from scipy import interpolate
importlib.reload(tcm)
n_vals = np.arange(30, 601, 1)
file_path = 'models/bbeta_carbon_te100ne-2.txt'
data = np.loadtxt(file_path)

#print(type(data[:, 0]))
n_spaced = data[:, 0]
tauC_spaced = data[:, 1]
#n_spaced = np.concatenate((n_test[:170], data[:, 0]))
#bbeta_spaced = np.concatenate((tautest[:170], data[:, 1]))

#print(n_test[:170])
#bbeta_spaced = b_spaced * beta_n(b_spaced, n_spaced, Te)

model = interpolate.interp1d(n_spaced, tauC_spaced, fill_value="extrapolate")
bbeta_carbon = model(n_vals)
Te = 85
tau_model_carbon_2 = tcm.tau_test(Te, EM, n_vals, bbeta_carbon) #* np.exp(-tcm.tau_C(Te, n_vals, EM))
#plt.plot(n_vals, tau_model_carbon, label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
plt.plot(n_vals, tau_model_carbon_2, label = r'Carbon', ls = '-', c = "#000000", lw = 2.5)
plt.axvline(285)
plt.plot(n_vals[85:255], tau_model_carbon_2[85:255])
plt.axhline(0)

In [ ]:
from scipy import interpolate
n_vals = np.arange(30, 601, 1)
file_path = 'models/inttau_carbon_te85_ne004.txt'
data = np.loadtxt(file_path)

#print(type(data[:, 0]))
n_spaced = data[:, 0]
tauC_spaced = data[:, 1]
#n_spaced = np.concatenate((n_test[:170], data[:, 0]))
#bbeta_spaced = np.concatenate((tautest[:170], data[:, 1]))

#print(n_test[:170])
#bbeta_spaced = b_spaced * beta_n(b_spaced, n_spaced, Te)

model = interpolate.interp1d(n_spaced, tauC_spaced, fill_value="extrapolate")
tau_model_carbon = model(n_vals)


#bbeta_vals = b_vals * beta_n(b_vals, n_vals, Te)
#plt.plot(n_test[:170], tautest[:170], label = r'Carbon $\tau_{RRL}$')
#plt.axhline(-5)
#plt.scatter(n_vals, tau_model_carbon, label = r'Carbon $\tau_{RRL}$')
#EM = 0.02
#Te = 85
#test_tau = tcm.tau_test(Te, EM, n_vals, tau_model_carbon) * np.exp(-tcm.tau_C(Te, n_vals, EM))
#plt.plot(n_vals, tau_model_carbon, label = r'Carbon $\int \tau_{RRL} e^{-\tau_C} d\nu$')

tau_array2 = [-8.54, -8.27, -8.48, -7.6]
n_array2 = [260, 266, 271, 276]
plt.scatter(n_array2, tau_array2)
#doi:10.1093/mnras/stw2818
tau_array = [5.63, 5.58, 6.15, 6.61, 6.81, 7.3, 7.47, 7.97, 8.82, 8.74, 7.95, 9.05]
n_array = [438, 448, 459, 467, 477, 485, 496, 510, 527, 542, 559, 575]
plt.scatter(n_array, np.array(tau_array))

#plt.plot(n_vals, tau_model_carbon_2, label = r'Carbon', ls = '-', c = "#FF0000", lw = 2.5)
plt.plot(n_vals[:170], tau_model_carbon_2[85:255], label = r'Carbon', ls = '-', c = "#FF0000", lw = 2.5)
#plt.plot(n_vals[:316], tau_model_carbon_2[255:], label = r'Carbon', ls = '-', c = "#FF0000", lw = 2.5)
plt.plot(n_spaced, tauC_spaced, label = r'Carbon $\int \tau_{RRL} e^{-\tau_C} d\nu$')
full_tau_combined = np.concatenate((tau_model_carbon_2[85:255], tauC_spaced))
full_n_combined = np.concatenate((n_vals[:170], n_spaced))
model_2_carbon = interpolate.interp1d(full_n_combined, full_tau_combined, fill_value="extrapolate")
full_carbon_interp = model_2_carbon(n_vals)
plt.plot(n_vals, full_carbon_interp, label = r'Carbon $\int \tau_{RRL} e^{-\tau_C} d\nu$', ls = '-', c = "#FF0000", lw = 2.5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate

n_vals = np.arange(30, 601, 1)

file_path = 'models/inttau_carbon.txt'
data = np.loadtxt(file_path)

n_high = data[:, 0]
tau_high = data[:, 1]

# --- join point ---
n_join = 210

# left-side grid: everything before n=200
mask_low = n_test < n_join
n_low = n_test[mask_low]

# interpolation of your original low-n model
low_raw_model = interpolate.interp1d(
    n_test,
    tautest,
    fill_value="extrapolate",
    bounds_error=False
)

# high-n value at the join
tau_high_join = np.interp(n_join, n_high, tau_high)

# --- tunable matching parameters ---
delta_n = 25      # positive shifts the left feature to smaller plotted n
amp = 0.5        # <1 flattens the low-n branch
baseline = np.median(tautest[n_test < 120])  # or just set baseline = 0.0

# shifted + flattened low branch
tau_low_shift_flat = baseline + amp * (
    low_raw_model(n_low + delta_n) - baseline
)

# force the transformed left branch to hit the high branch at n=200
tau_low_join_virtual = baseline + amp * (
    low_raw_model(n_join + delta_n) - baseline
)

offset = tau_high_join - tau_low_join_virtual

tau_low_matched = tau_low_shift_flat + offset

# concatenate transformed low branch with paper branch
n_spaced = np.concatenate((n_low, n_high))
bbeta_spaced = np.concatenate((tau_low_matched, tau_high))

# final interpolation
model = interpolate.interp1d(
    n_spaced,
    bbeta_spaced,
    fill_value="extrapolate",
    bounds_error=False
)

tau_model_carbon = model(n_vals)

plt.plot(n_vals, tau_model_carbon, label=r'Carbon $\tau_{\rm RRL}$')
plt.axvline(n_join, ls='--', alpha=0.5)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

data = np.loadtxt("models/bbeta_carbon.txt")

n_spaced = data[:, 0]
bbeta_spaced = data[:, 1]

#n_spaced = n_vals
#bbeta_spaced = tau_model_carbon


sort_id = np.argsort(n_spaced)
n_spaced = n_spaced[sort_id]
bbeta_spaced = bbeta_spaced[sort_id]

def asym_gaussian(n, amp, center, sig_left, sig_right):
    sig = np.where(n < center, sig_left, sig_right)
    return amp * np.exp(-0.5 * ((n - center) / sig)**2)

def carbon_bbeta_model(n, C, Apos, npos, sigpos, Aneg, nneg, sigleft, sigright):
    pos = Apos * np.exp(-0.5 * ((n - npos) / sigpos)**2)
    neg = asym_gaussian(n, Aneg, nneg, sigleft, sigright)
    return C + pos - neg

# Rough initial guess; tune based on your digitized points
p0 = [
    0.0,    # C
    8.0,    # Apos
    170.0,  # npos
    35.0,   # sigpos
    28.0,   # Aneg
    320.0,  # nneg
    50.0,   # sigleft: sharp left side
    250.0   # sigright: broad right tail
]

bounds = (
    [-10, 6,   120, 10,  26,   220, 20,  80],
    [ 10, 15,  220, 90, 40,  600, 120, 600]
)
sigma = np.ones_like(bbeta_spaced)

# Care most about the trough
trough_mask = (n_spaced > 240) & (n_spaced < 380)
sigma[trough_mask] = 0.15

# Some weight on the positive peak
peak_mask = (n_spaced > 120) & (n_spaced < 230)
sigma[peak_mask] = 0.35

# Do not let the right tail dominate
tail_mask = n_spaced > 500
sigma[tail_mask] = 5.0

popt, pcov = curve_fit(
    carbon_bbeta_model,
    n_spaced,
    bbeta_spaced,
    p0=p0,
    bounds=bounds,
    sigma=sigma,
    absolute_sigma=False,
    maxfev=10000
)

names = ["C", "Apos", "npos", "sigpos", "Aneg", "nneg", "signeg"]

for name, val, lo, hi in zip(names, popt, bounds[0], bounds[1]):
    print(f"{name:7s} = {val:8.3f}   bounds=({lo}, {hi})")

i_min = np.argmin(bbeta_spaced)

print("Digitized minimum:")
print("n =", n_spaced[i_min])
print("bbeta =", bbeta_spaced[i_min])

n_plot = np.linspace(50, 1800, 3000)
y_plot = carbon_bbeta_model(n_plot, *popt)

i_model_min = np.argmin(y_plot)

print("Model minimum:")
print("n =", n_plot[i_model_min])
print("bbeta =", y_plot[i_model_min])

bbeta_vals = carbon_bbeta_model(n_vals, *popt)

n_plot = np.linspace(50, 1800, 1000)
plt.plot(n_plot, carbon_bbeta_model(n_plot, *popt),
         label="Carbon smooth approximation", c="#000000", lw=2.5)
plt.scatter(n_spaced, bbeta_spaced, c="#000000", s=30,
            label="Digitized points")
plt.axhline(0, c="0.5", lw=1)
plt.legend()